# Notebook de travail

Projet : Smart City Energy Forecasting — Tetouan


Importation des bibliothèques nécessaires

In [14]:
import pandas as pd
import numpy as np
from pathlib import Path

#Affichage complet pour l'audit
pd.set_option('display.max_columns', None)

Définition des chemins et de la configuration de renommage

In [15]:
pd.set_option('display.max_columns', None)

DATA_PATH = Path('../data/raw/Tetuan City power consumption.csv')

COLUMN_MAPPING = {
    "DateTime": "datetime",
    "Temperature": "temperature",
    "Humidity": "humidity",
    "Wind Speed": "wind_speed",
    "General diffuse flows": "general_diffuse_flows",
    "Diffuse flows": "diffuse_flows",
    "Zone 1 Power Consumption": "zone1_power",
    "Zone 2  Power Consumption": "zone2_power",
    "Zone 3  Power Consumption": "zone3_power"
}


Fonction de chargement et de préparation initiale


In [16]:
def load_prepare_data(filepath, column_mapping):
    """
    Charge le dataset, renomme les colonnes, configure l'index temporel
    et prépare les variables cibles.
    """
    df = pd.read_csv(filepath)
    df = df.rename(columns=column_mapping)

    # Conversion stricte avec format explicite (évite UserWarning)
    df["datetime"] = pd.to_datetime(
        df["datetime"], format="%Y-%m-%d %H:%M:%S", errors="coerce"
    )
    df = df.sort_values("datetime").reset_index(drop=True)
    df = df.drop_duplicates(subset="datetime", keep="first")
    df = df.set_index("datetime")

    # Variables cibles
    df["target"] = df["zone1_power"]
    df["total_load"] = df["zone1_power"] + df["zone2_power"] + df["zone3_power"]

    return df

data = load_prepare_data(DATA_PATH, COLUMN_MAPPING)
print(f"Dimensions du dataset : {data.shape}")
display(data.head())


Dimensions du dataset : (1, 10)


,temperature,humidity,wind_speed,general diffuse flows,diffuse flows,zone1_power,zone2_power,zone3_power,target,total_load
datetime,,,,,,,,,,
NaT,6.559,73.8,0.083,0.051,0.119,34055.6962,16128.87538,20240.96386,34055.6962,70425.53544


Fonction d'audit de la qualité des données

In [17]:
def audit_data_quality(df):
    """
    Génère un tableau de synthèse sur les types, valeurs manquantes 
    et statistiques descriptives basiques.
    """
    audit = pd.DataFrame({
        "Type": df.dtypes.astype(str),
        "Valeurs Manquantes": df.isna().sum(),
        "Taux de Manquants (%)": (df.isna().mean() * 100).round(2)
    })

    # Ajout des statistiques descriptives
    summary = df.describe().T
    audit = audit.join(summary[["min", "mean", "max"]], how="left")

    # Vérification de la fréquence temporelle (robuste aux petits jeux de données)
    try:
        idx = pd.DatetimeIndex(df.index) if not isinstance(df.index, pd.DatetimeIndex) else df.index
        valid_idx = idx.dropna().unique()
        if len(valid_idx) >= 3:
            inferred_freq = pd.infer_freq(valid_idx[:10])
        else:
            inferred_freq = None
    except Exception:
        inferred_freq = None

    print(f"Fréquence temporelle inférée sur les 10 premières lignes : {inferred_freq}")
    if len(df.index.dropna()) > 0:
        print(f"Période couverte : du {df.index.min()} au {df.index.max()}\n")
    else:
        print("Période couverte : index temporel vide\n")

    return audit

# Exécution de l'audit
audit_results = audit_data_quality(data)
display(audit_results)

Fréquence temporelle inférée sur les 10 premières lignes : None
Période couverte : index temporel vide



,Type,Valeurs Manquantes,Taux de Manquants (%),min,mean,max
temperature,float64,0,0.0,6.55900,6.55900,6.55900
humidity,float64,0,0.0,73.80000,73.80000,73.80000
wind_speed,float64,0,0.0,0.08300,0.08300,0.08300
general diffuse flows,float64,0,0.0,0.05100,0.05100,0.05100
diffuse flows,float64,0,0.0,0.11900,0.11900,0.11900
zone1_power,float64,0,0.0,34055.69620,34055.69620,34055.69620
zone2_power,float64,0,0.0,16128.87538,16128.87538,16128.87538
zone3_power,float64,0,0.0,20240.96386,20240.96386,20240.96386
target,float64,0,0.0,34055.69620,34055.69620,34055.69620
total_load,float64,0,0.0,70425.53544,70425.53544,70425.53544
